In [ ]:
import numpy as np
from tqdm import tqdm

# 计算杰卡德相似性矩阵
def Jaccard_similarity(A):
    # B是返回的杰卡德相似性矩阵
    B = np.zeros((A.shape[0], A.shape[0]))
    for i in tqdm(range(B.shape[0])):
        for j in range(i + 1, B.shape[1]):  # 只做上三角部分
            if np.sum(A[i]) == 0 and np.sum(A[j]) == 0:  # 如果两个药物不和任何靶点相互作用，则两个药物的相似度为0
                B[i][j] = 0
            else:
                jiaoji = 0
                bingji = 0
                # 计算A[i]和A[j]的交集和并集
                for k in range(A.shape[1]):
                    if A[i][k] == 1 and A[j][k] == 1:
                        jiaoji += 1
                        bingji += 1
                    elif A[i][k] == 1 or A[j][k] == 1:
                        bingji += 1
                B[i][j] = jiaoji / bingji

    row, col = np.diag_indices_from(B)
    B[row, col] = 1
    B += B.T - np.diag(B.diagonal())

    return B

# 计算相似性矩阵的熵值
def calculate_entropy(matrix):
    """
    计算相似性矩阵的熵值
    :param matrix: 相似性矩阵 (n x n)
    :return: 熵值 (标量)
    """
    # 归一化矩阵，确保每行的和为1
    row_sums = matrix.sum(axis=1, keepdims=True)
    normalized_matrix = matrix / row_sums

    # 计算熵值
    entropy = -np.sum(normalized_matrix * np.log(normalized_matrix + 1e-10), axis=1)  # 加1e-10避免log(0)
    return entropy.mean()  # 返回平均熵值

# 基于熵权法计算权重
def entropy_weight(similarity_matrices):
    """
    基于熵权法计算权重
    :param similarity_matrices: 多个相似性矩阵的列表 [matrix1, matrix2, ...]
    :return: 每个相似性矩阵的权重列表 [weight1, weight2, ...]
    """
    # 计算每个相似性矩阵的熵值
    entropies = [calculate_entropy(matrix) for matrix in similarity_matrices]

    # 计算权重
    weights = []
    for entropy in entropies:
        if entropy == 0:  # 如果熵为0，直接赋权重为0
            weights.append(0)
        else:
            weights.append(1 / entropy)  # 权重与熵成反比

    # 归一化权重
    weights = np.array(weights) / np.sum(weights)

    return weights

if __name__ == "__main__":
    # 加载药物相关的矩阵数据
    drug_drug_interaction = np.loadtxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/mat_drug_drug.txt')  # 药物-药物相互作用矩阵
    drug_disease_association = np.loadtxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/mat_drug_disease.txt')
    drug_sideeffect_association = np.loadtxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/mat_drug_se.txt')
    drug_drug_chemistry_similarity = np.loadtxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/Similarity_Matrix_Drugs.txt')

    # 计算杰卡德相似性矩阵
    drug_drug_interaction_similarity = Jaccard_similarity(drug_drug_interaction)
    drug_disease_association_similarity = Jaccard_similarity(drug_disease_association)
    drug_sideeffect_association_similarity = Jaccard_similarity(drug_sideeffect_association)

    # 保存杰卡德相似性矩阵
    np.save('/home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_drug_interaction_similarity.npy', drug_drug_interaction_similarity)
    np.save('/home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_disease_association_similarity.npy', drug_disease_association_similarity)
    np.save('/home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_sideeffect_association_similarity.npy', drug_sideeffect_association_similarity)
    np.save('/home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_drug_chemistry_similarity.npy', drug_drug_chemistry_similarity)

    # 将相似性矩阵放入列表
    similarity_matrices = [
        drug_drug_interaction_similarity,
        drug_disease_association_similarity,
        drug_sideeffect_association_similarity,
        drug_drug_chemistry_similarity
    ]

    # 计算权重
    weights = entropy_weight(similarity_matrices)

    # 输出权重结果
    print("权重分配结果：")
    for i, weight in enumerate(weights):
        print(f"相似性矩阵 {i + 1} 的权重: {weight:.4f}")

    # 使用权重进行加权融合
    drug_fusion_similarity = np.zeros_like(similarity_matrices[0])
    for matrix, weight in zip(similarity_matrices, weights):
        drug_fusion_similarity += weight * matrix

    # 保存融合后的相似性矩阵
    np.savetxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_fusion_similarity_708_708.txt', drug_fusion_similarity)

    # 加载靶点相关的矩阵数据
    target_disease_association = np.loadtxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/mat_protein_disease.txt')
    target_target_interaction = np.loadtxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/mat_protein_protein.txt')
    target_target_sequence_similarity = np.loadtxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/Similarity_Matrix_Proteins.txt')
    #target_complex_similarity = np.load('/home/ubuntu/data/xjh/xlw/DTINet-master/data/protein_complex_similarity_matrix.npy')

    # 计算杰卡德相似性矩阵
    target_disease_association_similarity = Jaccard_similarity(target_disease_association)
    target_target_interaction_similarity = Jaccard_similarity(target_target_interaction)

    # 保存杰卡德相似性矩阵
    np.save('/home/ubuntu/data/xjh/xlw/DTINet-master/data/target_disease_association_similarity.npy', target_disease_association_similarity)
    np.save('/home/ubuntu/data/xjh/xlw/DTINet-master/data/target_target_interaction_similarity.npy', target_target_interaction_similarity)
    np.save('/home/ubuntu/data/xjh/xlw/DTINet-master/data/target_target_sequence_similarity.npy', target_target_sequence_similarity)

    # 将相似性矩阵放入列表
    target_similarity_matrices = [
        target_disease_association_similarity,
        target_target_interaction_similarity,
        target_target_sequence_similarity
    ]

    # 计算权重
    target_weights = entropy_weight(target_similarity_matrices)

    # 输出权重结果
    print("靶点相似性矩阵权重分配结果：")
    for i, weight in enumerate(target_weights):
        print(f"靶点相似性矩阵 {i + 1} 的权重: {weight:.4f}")

    # 使用权重进行加权融合
    target_fusion_similarity = np.zeros_like(target_similarity_matrices[0])
    for matrix, weight in zip(target_similarity_matrices, target_weights):
        target_fusion_similarity += weight * matrix

    # 保存融合后的相似性矩阵
    np.savetxt('/home/ubuntu/data/xjh/xlw/DTINet-master/data/target_fusion_similarity_3_1512_1512.txt', target_fusion_similarity)

    print('end')

100%|██████████| 708/708 [13:27<00:00,  1.14s/it]


权重分配结果：
相似性矩阵 1 的权重: 0.3635
相似性矩阵 2 的权重: 0.2195
相似性矩阵 3 的权重: 0.2096
相似性矩阵 4 的权重: 0.2075


100%|██████████| 1512/1512 [21:48<00:00,  1.16it/s]


靶点相似性矩阵权重分配结果：
靶点相似性矩阵 1 的权重: 0.1929
靶点相似性矩阵 2 的权重: 0.6146
靶点相似性矩阵 3 的权重: 0.1925
end


In [1]:
import pandas as pd

# ========== 你的文件路径，无需修改 ==========
txt_path = "/home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_fusion_similarity_708_708.txt"
csv_path = "/home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_fusion_similarity_708_708.csv"

# 读取txt矩阵 + 转csv，自动处理 多个空格/单个空格 分隔，完美适配
df = pd.read_csv(txt_path, sep="\s+", header=None, index_col=None)
# 保存为csv，不生成多余索引，纯数值矩阵，和原txt完全对应
df.to_csv(csv_path, index=False, header=False)

print("转换完成！CSV文件路径：", csv_path)

转换完成！CSV文件路径： /home/ubuntu/data/xjh/xlw/DTINet-master/data/drug_fusion_similarity_708_708.csv
